# Automated Cryptocurrency Trading using Deep RL
**Normalisation: Rolling Z-Score** — each price column z-scored over the 24-bar observation window → adapts to any local price regime

**Double Dueling DQN with Prioritized Experience Replay**

### Setup Instructions
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run cells in order
3. In Cell 2, upload your 4 HDF5 data files when prompted

In [ ]:
# Remove TensorFlow and JAX — not needed for PyTorch RL, but their presence
# causes a tensorboard→TF→JAX circular import that crashes 'import tianshou'
!pip uninstall -q -y tensorflow tensorflow-cpu tensorflow-gpu keras tf-keras jax jaxlib 2>/dev/null; echo 'TF/JAX removed'

# Install tianshou WITHOUT torch dependency to keep Kaggle's pre-installed CUDA torch
!pip install -q tianshou==0.4.11 --no-deps
# Install tianshou's other dependencies manually (excluding torch)
!pip install -q gym==0.25.2 "numpy<2.0" wandb seaborn matplotlib pandas tables tqdm ccxt tensorboard numba h5py packaging
print('Installation done. Restarting kernel to apply numpy downgrade...')
import os
os.kill(os.getpid(), 9)

In [ ]:
import numpy as np
import gym
print(f'numpy version: {np.__version__}')
print(f'gym version: {gym.__version__}')
print('All good, continue running the remaining cells.')

In [ ]:
import os, shutil, glob

KAGGLE_INPUT = '/kaggle/input'
DATA_DIR = 'Data'
os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(KAGGLE_INPUT):
    print('Scanning /kaggle/input...')
    all_files = glob.glob(f'{KAGGLE_INPUT}/**/*', recursive=True)
    for f in all_files:
        print(f)
    hdf5_files = glob.glob(f'{KAGGLE_INPUT}/**/*.hdf5', recursive=True)
    if not hdf5_files:
        raise FileNotFoundError('No HDF5 files found. Check printed paths above.')
    for src in hdf5_files:
        dst = os.path.join(DATA_DIR, os.path.basename(src))
        shutil.copy(src, dst)
        print(f'Copied: {src} -> {dst}')
else:
    from google.colab import files
    print('Upload your 4 HDF5 files:')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, f'{DATA_DIR}/{fname}')

print('Files in Data/:', os.listdir(DATA_DIR))

In [ ]:
import wandb
wandb.login()

## Neural Network (trading_network_utilities.py)

In [ ]:
"""
Automatic Cryptocurrency trading using Deep RL
Nick Kaparinos
2022
"""

import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import math


class MLP(nn.Module):
    """ Q and V Multi Layer Perceptron (MLP) networks with MLP/LSTM/CNN/Attention timeseries encoders """

    def __init__(self, state_shape, action_shape, n_features, n_previous_timesteps, n_timeseries, encoder_type='LSTM',
                 n_neurons=128, encoder_n_linear_layers=2, q_n_linear_layers=2, v_n_linear_layers=2, n_posmlp_layers=2,
                 n_head_layers=2, n_attention_blocks=1, dueling=True, n_cnn_layers=2, device='cpu') -> None:
        assert q_n_linear_layers >= 1 and v_n_linear_layers >= 1
        super().__init__()
        self.encoder_type = encoder_type
        self.n_features = n_features
        self.n_previous_timesteps = n_previous_timesteps
        self.n_timeseries = n_timeseries
        self.device = device
        self.dueling = dueling
        self.action_dim = int(np.prod(action_shape))

        # TimeseriesEncoder
        self.encoder = TimeseriesEncoder(n_features, n_previous_timesteps, n_timeseries, encoder_type, n_neurons,
                                         encoder_n_linear_layers, n_posmlp_layers, n_head_layers, n_attention_blocks,
                                         n_cnn_layers, device)

        # Q, V Networks
        self.q_network = []
        for i in range(q_n_linear_layers):
            input_dim = n_neurons if i != 0 else self.n_timeseries * n_neurons + self.n_timeseries + 1
            output_dim = n_neurons if i != q_n_linear_layers - 1 else self.action_dim
            self.q_network.append(nn.Linear(input_dim, output_dim))

            if i != q_n_linear_layers - 1:
                self.q_network.append(nn.ReLU())
        self.q_network = nn.Sequential(*self.q_network)

        self.v_network = []
        for i in range(v_n_linear_layers):
            input_dim = n_neurons if i != 0 else self.n_timeseries * n_neurons + self.n_timeseries + 1
            output_dim = n_neurons if i != v_n_linear_layers - 1 else 1
            self.v_network.append(nn.Linear(input_dim, output_dim))

            if i != v_n_linear_layers - 1:
                self.v_network.append(nn.ReLU())
        self.v_network = nn.Sequential(*self.v_network)

    def forward(self, s, state, info):
        """ Mapping: s -> logits. """
        # Timeseries Embeddings
        portfolio_state = torch.as_tensor(s[:, -(self.n_timeseries + 1):], device=self.device, dtype=torch.float32)
        s = torch.as_tensor(s[:, :-(self.n_timeseries + 1)].reshape(s.shape[0], self.n_previous_timesteps + 1, -1),
                            device=self.device, dtype=torch.float32)  # type: ignore
        timeseries_embeddings = self.encoder(s)

        # Heads
        input_features = torch.hstack([portfolio_state, timeseries_embeddings])
        q = self.q_network(input_features)
        if self.dueling:
            v = self.v_network(input_features)
            logits = q - q.mean(dim=1, keepdim=True) + v
        else:
            logits = q
        return logits, state


class TimeseriesEncoder(nn.Module):
    """ Encoder networks for each timeseries """

    def __init__(self, n_features, n_previous_timesteps, n_timeseries, encoder_type='LSTM', n_neurons=128,
                 encoder_n_linear_layers=2, n_posmlp_layers=2, n_head_layers=2, n_attention_blocks=1, n_cnn_layers=2,
                 device='cpu') -> None:
        super().__init__()
        self.n_features = n_features
        self.n_previous_timesteps = n_previous_timesteps
        self.n_timeseries = n_timeseries
        self.device = device

        # Encoder network for each timeseries
        self.encoder = nn.ModuleList()
        for i in range(n_timeseries):
            if encoder_type == 'LSTM':
                self.encoder += [LSTMWrapper(n_features, n_neurons)]
            elif encoder_type == 'MLP':
                self.encoder = [nn.Flatten(), nn.Linear(self.n_features * (self.n_previous_timesteps + 1), n_neurons),
                                nn.ReLU()]
                for _ in range(encoder_n_linear_layers - 1):
                    self.encoder.extend([nn.Linear(n_neurons, n_neurons), nn.ReLU()])
                self.encoder += [nn.Sequential(*self.encoder)]
            elif encoder_type == 'Attention':
                self.encoder += [AttentionEncoder(n_features, n_previous_timesteps, n_neurons, n_posmlp_layers,
                                                  n_head_layers, n_attention_blocks)]
            elif encoder_type == 'CNN':
                self.encoder += [CNNEncoder(n_features, n_previous_timesteps, n_neurons, n_cnn_layers)]
            else:
                raise ValueError(f'TimeseriesEncoder type {encoder_type} not supported!')

    def forward(self, x):
        """ Encoder each timeseries and return the embeddings """
        timeseries_embeddings = torch.empty(0, device=self.device, dtype=torch.float32)
        for i in range(self.n_timeseries):
            temp_embeddings = self.encoder[i](x[:, :, self.n_features * i:self.n_features * (i + 1)])  # noqa
            timeseries_embeddings = torch.cat((timeseries_embeddings, temp_embeddings), dim=1)

        return timeseries_embeddings


class AttentionEncoder(nn.Module):
    """ Attention Encoder consisting of Attention Block(s) and an MLP head """

    def __init__(self, n_features, n_previous_timesteps, n_neurons=128, n_posmlp_layers=2, n_head_layers=2,
                 n_attention_blocks=1) -> None:
        assert n_head_layers >= 1
        super().__init__()
        self.attention_blocks = [AttentionBlock(n_features, n_previous_timesteps, n_neurons, n_posmlp_layers) for _ in
                                 range(n_attention_blocks)]
        self.attention_blocks = nn.Sequential(*self.attention_blocks)

        self.mlp_head = nn.ModuleList([nn.Flatten(),
                                       nn.Linear(n_features * (n_previous_timesteps + 1), n_neurons),
                                       nn.ReLU()])
        for _ in range(n_head_layers - 1):
            self.mlp_head.extend([nn.Linear(n_neurons, n_neurons), nn.ReLU()])
        self.mlp_head = nn.Sequential(*self.mlp_head)
        self.pos_encoder = PositionalEncoding(n_features)

    def forward(self, x):
        x = self.pos_encoder(x)
        x = self.attention_blocks(x)
        x = self.mlp_head(x)
        return x


class AttentionBlock(nn.Module):
    """ Attention Encoder block consisting of Multi head Attention, LayerNorm and Position Wise MLP network
    https://arxiv.org/abs/1706.03762
    """

    def __init__(self, n_features, n_previous_timesteps, n_neurons=128, n_posmlp_layers=2) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(n_features, 1)
        self.norm1 = nn.LayerNorm([n_previous_timesteps + 1, n_features])
        self.norm2 = nn.LayerNorm([n_previous_timesteps + 1, n_features])
        self.position_wise_mlp = PositionWiseMLP(n_features, n_previous_timesteps, n_neurons, n_posmlp_layers)

    def forward(self, x):
        x = self.norm1(x + self.attn(x, x, x)[0])
        x = self.norm2(x + self.position_wise_mlp(x))
        return x


class PositionWiseMLP(nn.Module):
    """ Position wise MLP network used in transformer block """

    def __init__(self, n_features, n_previous_timesteps, n_neurons=128, n_posmlp_layers=2) -> None:
        assert n_posmlp_layers >= 2
        super().__init__()
        self.n_previous_timesteps = n_previous_timesteps
        self.position_wise_mlp = nn.ModuleList()

        for _ in range(n_previous_timesteps + 1):
            temp_mlp = nn.ModuleList([nn.Linear(n_features, n_neurons), nn.ReLU()])
            for __ in range(n_posmlp_layers - 2):
                temp_mlp.extend([nn.Linear(n_neurons, n_neurons), nn.ReLU()])
            temp_mlp.extend([nn.Linear(n_neurons, n_features), nn.ReLU()])

            self.position_wise_mlp.append(nn.Sequential(*temp_mlp))

    def forward(self, x):
        y = torch.empty(0, dtype=torch.float32)
        for i in range(self.n_previous_timesteps + 1):
            output = self.position_wise_mlp[i](x[:, i, :])
            output = output[:, None, :]
            if i == 0:
                y = output
            else:
                y = torch.cat((y, output), dim=1)
        return y


class LSTMWrapper(nn.Module):
    """ Simple LSTM wrapper class """

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=False)

    def forward(self, x):
        return self.lstm(x)[0][:, -1, :]


class CNNEncoder(nn.Module):
    """ 1D Convolutional timeseries encoder """

    def __init__(self, n_features, n_previous_timesteps, n_neurons=128, n_cnn_layers=2):
        super().__init__()
        assert n_cnn_layers >= 2
        self.encoder = [nn.Conv1d(n_features, n_neurons, kernel_size=2), nn.ReLU()]

        for i in range(n_cnn_layers - 1):
            self.encoder.extend([nn.Conv1d(n_neurons, n_neurons, kernel_size=2), nn.ReLU()])
        self.encoder = nn.Sequential(*self.encoder)

        self.encoder_head = nn.Linear((n_previous_timesteps + 1 - n_cnn_layers) * n_neurons, n_neurons)

    def forward(self, x):
        x = self.encoder(x.permute(0, 2, 1))
        x = self.encoder_head(x.view(x.shape[0], -1))
        return x


class PositionalEncoding(nn.Module):
    """ Positional encoding class """

    def __init__(self, d_model: int, max_len: int = 200):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [seq_len, batch_size, embedding_dim]
        """
        x = x.permute(1, 0, 2)
        x = x + self.pe[:x.size(0)]
        x = x.permute(1, 0, 2)
        return x


class Actor(nn.Module):
    def __init__(self, net, use_softmax=False):
        super(Actor, self).__init__()
        self.net = net
        self.use_softmax = use_softmax

    def forward(self, obs, state=None, info={}):
        x, state = self.net(obs, state, info)
        if self.use_softmax:
            x = F.softmax(x, dim=-1)
        return x, state


class Critic(nn.Module):
    def __init__(self, net):
        super(Critic, self).__init__()
        self.net = net

    def forward(self, obs, act=None, info={}):
        x, state = self.net(obs, None, info)
        return x

print('trading_network_utilities loaded.')


## Trading Environment (trading_utilities.py)

In [ ]:
"""
Automatic Cryptocurrency trading using Deep RL
Nick Kaparinos
2022
"""

import gym
import numpy as np
from os import makedirs
import pandas as pd
import torch
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
import seaborn as sns


class TradeEnv(gym.Env):
    """ Crypto trading environment """
    reward_range = (-float('inf'), float('inf'))
    metadata = {'render.modes': None}

    def __init__(self, crypto_files=(), timeseries_step='m', test=False, n_previous_timesteps=5, max_episode_steps=500):
        super().__init__()
        self.n_timeseries = len(crypto_files)
        self.n_features = 6
        self.n_previous_timesteps = n_previous_timesteps
        self.observation_space = gym.spaces.Discrete(
            (1 + self.n_previous_timesteps) * (self.n_features * self.n_timeseries) + self.n_timeseries + 1)
        self.action_space = gym.spaces.Discrete(self.n_timeseries + 1)
        self.max_episode_steps = max_episode_steps
        self.current_step = None
        self.episode_starting_step = None
        self.timeseries_step = timeseries_step

        self.starting_balance = 1_000
        self.balance = self.starting_balance

        self.portfolio = np.zeros(self.n_timeseries)
        self.fee = 0.001

        # Read
        self.crypto_names = [crypto[5:8] for crypto in crypto_files]
        cryptos = [pd.read_hdf(crypto).resample(timeseries_step).agg(
            {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'}) for crypto in tqdm(crypto_files)]
        common_start, common_end = self._find_common_timespan(cryptos)
        for i in range(len(cryptos)):
            cryptos[i] = cryptos[i][common_start <= cryptos[i].index]
            cryptos[i] = cryptos[i][cryptos[i].index <= common_end]

        self.data = pd.DataFrame()
        for i in range(self.n_timeseries):
            cryptos[i]['month'] = cryptos[i].index.month
            cryptos[i]['day'] = cryptos[i].index.day
            cryptos[i]['hour'] = cryptos[i].index.hour
            cryptos[i].columns = [f'open{i}', f'high{i}', f'low{i}', f'close{i}', f'month{i}', f'day{i}', f'hour{i}']
            self.data = pd.concat([self.data, cryptos[i]], axis=1)
        self.data = self.data.interpolate()

        # Split train test
        open_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                'open' in i]
        n = self.data.shape[0]
        self.means = []
        self.stds = []
        for idx, i in enumerate(open_columns_numbers):
            self.means.append(self.data.iloc[:int(n * 0.8), i].mean())
            self.stds.append(self.data.iloc[:int(n * 0.8), i].std())
        if test:
            self.data = self.data.iloc[int(n * 0.8):]
        else:
            self.data = self.data.iloc[:int(n * 0.8)]

    def step(self, action):
        open_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                'open' in i]
        close_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                 'close' in i]
        open_stock_prices = self.data.iloc[
            self.episode_starting_step + self.current_step + 1, open_columns_numbers].values
        close_stock_prices = self.data.iloc[
            self.episode_starting_step + self.current_step + 1, close_columns_numbers].values
        episode_starting_portfolio_value = self.balance + np.inner(self.portfolio, open_stock_prices)

        if action == self.action_space.n - 1:  # sell all
            self.balance += np.inner(self.portfolio, open_stock_prices)
            self.portfolio = np.zeros(self.n_timeseries)
        elif self.portfolio[action] == 0 or self.balance != 0:  # Sell portfolio and buy, otherwise hold
            self.balance += np.inner(self.portfolio, open_stock_prices)
            self.portfolio = np.zeros(self.n_timeseries)
            stock_bought = self.balance / (open_stock_prices[action] * (1 + self.fee))
            self.portfolio[action] = stock_bought
            self.balance = 0

        self.current_step += 1
        if self.current_step == self.max_episode_steps:
            done = True
            obs = np.zeros(self.observation_space.n)
        else:
            done = False
            obs = self._get_obs(open_columns_numbers)

        episode_ending_portfolio_value = self.balance + np.inner(self.portfolio, close_stock_prices)
        reward = episode_ending_portfolio_value - episode_starting_portfolio_value

        return obs, reward, done, {}

    def seed(self, seed=None):
        return seed

    def reset(self):
        self.episode_starting_step = random.randint(self.n_previous_timesteps,
                                                    self.data.shape[0] - self.max_episode_steps - 1)
        self.portfolio = [0] * self.n_timeseries
        self.balance = self.starting_balance
        self.current_step = 0

        open_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                'open' in i]
        obs = self._get_obs(open_columns_numbers)
        return obs

    def _get_obs(self, open_columns_numbers):
        """ Get observation """
        all_feature_columns = sorted(set([i for i in range(self.data.shape[1])]) - set(open_columns_numbers))
        obs = np.empty((0, 0), float)

        for i in range(self.n_timeseries):
            feature_columns = [column for column in all_feature_columns if str(i) in self.data.columns[column]]
            timeseries_obs = self.data.iloc[
                             self.episode_starting_step + self.current_step - self.n_previous_timesteps:
                             self.episode_starting_step + self.current_step + 1,
                             feature_columns].values
            # Option 3: z-score each price column over the observation window
            for _col in range(3):
                _d = timeseries_obs[:, _col]
                _std = _d.std()
                timeseries_obs[:, _col] = (
                    (_d - _d.mean()) / _std if _std > 1e-8 else np.zeros_like(_d))
            timeseries_obs[:, 3] /= 12  # month
            timeseries_obs[:, 4] /= 31  # day
            timeseries_obs[:, 5] /= 24  # hour
            if obs.shape[0] == 0:
                obs = timeseries_obs.copy()
            else:
                obs = np.hstack([obs, timeseries_obs])
        portfolio_state1 = np.array(self.portfolio) > 0
        portfolio_state2 = np.array([np.logical_not(portfolio_state1.sum())])
        portfolio_state = np.concatenate((portfolio_state1, portfolio_state2)) * 1
        obs = np.concatenate([obs.ravel(), portfolio_state])
        return obs

    @staticmethod
    def _find_common_timespan(timeseries):
        starts = [i.index[0] for i in timeseries]
        ends = [i.index[-1] for i in timeseries]
        return max(starts), min(ends)


class TestTradeEnv(TradeEnv):
    """ Test environment for cryptocurrency trading
    Includes episode and epoch performance visualisations over base environment
    """

    def __init__(self, log_dir, num_test_episodes, **kwargs):
        super().__init__(test=True, **kwargs)
        self.log_dir = log_dir
        self.epoch = -1
        self.episode = num_test_episodes
        self.num_test_episodes = num_test_episodes

        self.actions_chosen = dict()
        self.reward_per_timeseries = np.zeros(self.n_timeseries)
        self.portfolio_value_list = []
        self.reward_list = []
        self.episode_ending_portfolio_value_list = []
        self.epoch_distributions = []
        self.epoch_versus_bah_distributions = []

    def step(self, action):
        open_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                'open' in i]
        close_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                 'close' in i]
        open_stock_prices = self.data.iloc[
            self.episode_starting_step + self.current_step + 1, open_columns_numbers].values
        close_stock_prices = self.data.iloc[
            self.episode_starting_step + self.current_step + 1, close_columns_numbers].values
        episode_starting_portfolio_value = self.balance + np.inner(self.portfolio, open_stock_prices)

        if action == self.action_space.n - 1:  # sell all
            self.balance += np.inner(self.portfolio, open_stock_prices)
            self.portfolio = np.zeros(self.n_timeseries)
            self.actions_chosen[self.current_step] = action
        elif self.portfolio[action] == 0 or self.balance != 0:  # otherwise, hold
            self.balance += np.inner(self.portfolio, open_stock_prices)
            self.portfolio = np.zeros(self.n_timeseries)
            stock_bought = self.balance / (open_stock_prices[action] * (1 + self.fee))
            self.portfolio[action] = stock_bought
            self.balance = 0

            self.actions_chosen[self.current_step] = action

        episode_ending_portfolio_value = self.balance + np.inner(self.portfolio, close_stock_prices)
        reward = episode_ending_portfolio_value - episode_starting_portfolio_value

        if action != self.action_space.n - 1:
            self.reward_per_timeseries[action] += reward
        self.portfolio_value_list.append(episode_ending_portfolio_value)
        self.reward_list.append(reward)

        self.current_step += 1
        if self.current_step == self.max_episode_steps:
            done = True
            obs = np.zeros(self.observation_space.n)

            self.episode_ending_portfolio_value_list.append(episode_ending_portfolio_value)
            self.episode_ending_value_versus_bah_list.append(episode_ending_portfolio_value / self.bah_ending_value)
            self._save_episode_results()
            self.episode += 1
        else:
            done = False
            obs = self._get_obs(open_columns_numbers)

        return obs, reward, done, {}

    def reset(self):
        self.actions_chosen = dict()
        self.reward_per_timeseries = np.zeros(self.n_timeseries)
        self.portfolio_value_list = []
        self.reward_list = []

        obs = super().reset()

        open_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                'open' in i]
        close_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                 'close' in i]
        starting_stock_prices = self.data.iloc[
            self.episode_starting_step, open_columns_numbers].values
        ending_stock_prices = self.data.iloc[
            self.episode_starting_step + self.max_episode_steps, close_columns_numbers].values
        self.bah_ending_value = (
                self.starting_balance / self.n_timeseries * (
                    ending_stock_prices / (starting_stock_prices * (1 + self.fee)))).sum()

        # Reset epoch episode counter
        if self.episode == self.num_test_episodes:
            self.episode = 0
            self.epoch += 1
            self.episode_ending_portfolio_value_list = []
            self.episode_ending_value_versus_bah_list = []
            makedirs(f'{self.log_dir}epoch-{self.epoch}', exist_ok=True)
        return obs

    def _save_episode_results(self):
        """ Plot timeseries and agent`s actions. Then save the figure """

        # Agents actions plot
        sns.set()
        self._calc_buy_and_sell_timesteps()
        open_columns_numbers = [list(self.data.columns.values).index(i) for i in self.data.columns.values if
                                'open' in i]
        plt.figure(0)
        plt.clf()

        fig, axs = plt.subplots(self.n_timeseries, figsize=(14, 10))
        plt.suptitle('Episode Agent`s actions Visualization')
        plt.xlabel('Time')
        for i in range(self.n_timeseries):
            labels = [None, None, None]
            if i == 0:
                labels = ['timeseries', 'buy', 'sell']
            timeseries = self.data.iloc[self.episode_starting_step:self.episode_starting_step + self.max_episode_steps,
                         open_columns_numbers[i]].to_frame()
            timeseries.columns = [f'open {self.crypto_names[i]}']
            sns.lineplot(x=timeseries.index, y=timeseries.iloc[:, 0], label=labels[0], zorder=1, ax=axs[i])
            sns.scatterplot(x=timeseries.index[self.buy_and_sell_timesteps[i]['buy']],
                            y=timeseries.iloc[self.buy_and_sell_timesteps[i]['buy'], 0], label=labels[1], s=40,
                            marker='^', zorder=2, ax=axs[i])
            sns.scatterplot(x=timeseries.index[self.buy_and_sell_timesteps[i]['sell']],
                            y=timeseries.iloc[self.buy_and_sell_timesteps[i]['sell'], 0], label=labels[2], s=35,
                            marker='v', zorder=3, ax=axs[i])
        plt.savefig(f'{self.log_dir}epoch-{self.epoch}/test_episode-{self.episode}-actions.png', dpi=300)

        # Portfolio value plot
        plt.figure(0)
        plt.clf()
        sns.lineplot(x=[i for i in range(len(self.portfolio_value_list))], y=self.portfolio_value_list)
        plt.title('Portfolio value')
        plt.xlabel('Time')
        plt.ylabel('Portfolio value')
        plt.savefig(f'{self.log_dir}epoch-{self.epoch}/test_episode-{self.episode}-portfolio.png', dpi=200)

        # Reward plot
        plt.figure(0)
        plt.clf()
        sns.lineplot(x=[i for i in range(len(self.reward_list))], y=self.reward_list)
        plt.title('Episode reward per timestep')
        plt.xlabel('Time step')
        plt.ylabel('Reward')
        plt.savefig(f'{self.log_dir}epoch-{self.epoch}/test_episode-{self.episode}-reward.png', dpi=200)

        # Reward per timeseries plot
        plt.figure(0)
        plt.clf()
        sns.barplot(x=[f'series {i}' for i in range(self.n_timeseries)], y=self.reward_per_timeseries)
        plt.title('Episode reward per timeseries')
        plt.xlabel('Time step')
        plt.ylabel('Reward')
        plt.savefig(f'{self.log_dir}epoch-{self.epoch}/test_episode-{self.episode}-reward-per-series.png', dpi=200)

        if self.episode == self.num_test_episodes - 1:  # Save epoch ending portfolio value distribution plot
            self.epoch_distributions.append(self.episode_ending_portfolio_value_list)
            self.epoch_versus_bah_distributions.append(self.episode_ending_value_versus_bah_list)

            plt.figure(0)
            plt.clf()
            ax = sns.boxplot(y=self.episode_ending_portfolio_value_list)
            plt.title('Test episode ending portfolio value distribution')
            plt.xlabel('Portfolio end value')
            plt.savefig(f'{self.log_dir}epoch-{self.epoch}/boxplot_distribution', dpi=100)

            plt.figure(0)
            plt.clf()
            ax = sns.boxplot(y=self.episode_ending_value_versus_bah_list)
            plt.title('Test episode ending portfolio value / b&h value distribution')
            plt.xlabel('Portfolio end value ratio')
            plt.savefig(f'{self.log_dir}epoch-{self.epoch}/boxplot_versus_bah_distribution', dpi=100)

            # Save epoch test episode portfolio ending values
            self.episode_ending_portfolio_value_list.sort()
            self.episode_ending_value_versus_bah_list.sort()
            save_list_to_txt(self.episode_ending_portfolio_value_list,
                             f'{self.log_dir}epoch-{self.epoch}/epoch_distribution.txt')
            save_list_to_txt(self.episode_ending_value_versus_bah_list,
                             f'{self.log_dir}epoch-{self.epoch}/epoch_versus_bah_distribution.txt')

            # Plot all epochs` distributions in one plot
            df = pd.DataFrame(data=self.epoch_distributions).T
            df.columns = [f'Epoch {i}' for i in range(len(self.epoch_distributions))]

            plt.figure(0)
            plt.clf()
            ax = sns.boxplot(data=df)
            plt.title('Test episode ending portfolio value distribution')
            plt.ylabel('Portfolio value')
            plt.savefig(f'{self.log_dir}epoch-{self.epoch}/epoch_boxplot_distribution', dpi=100)

            df = pd.DataFrame(data=self.epoch_versus_bah_distributions).T
            df.columns = [f'Epoch {i}' for i in range(len(self.epoch_versus_bah_distributions))]

            plt.figure(0)
            plt.clf()
            ax = sns.boxplot(data=df)
            plt.title('Test episode ending portfolio value / b&h value distribution')
            plt.ylabel('Portfolio value ratio')
            plt.savefig(f'{self.log_dir}epoch-{self.epoch}/epoch_bah_boxplot_distribution', dpi=100)

    def _calc_buy_and_sell_timesteps(self):
        """ Process self.actions_chosen. Calculate buy and sell time steps for each stock """
        self.buy_and_sell_timesteps = [dict() for _ in range(self.n_timeseries)]

        for i in range(self.n_timeseries):
            self.buy_and_sell_timesteps[i]['buy'] = []
            self.buy_and_sell_timesteps[i]['sell'] = []

        for i, (timestep, action) in enumerate(self.actions_chosen.items()):
            if i != 0:
                if previous_action != self.action_space.n - 1:  # noqa
                    self.buy_and_sell_timesteps[previous_action]['sell'].append(timestep)  # noqa
            if action != self.action_space.n - 1:
                self.buy_and_sell_timesteps[action]['buy'].append(timestep)
            previous_action = action


max_episode_steps = 1000
gym.envs.register(id='TradeEnv-v0', entry_point=TradeEnv,
                  max_episode_steps=max_episode_steps)
gym.envs.register(id='TestTradeEnv-v0', entry_point=TestTradeEnv,
                  max_episode_steps=max_episode_steps)

print('trading_utilities loaded.')


## Utilities (utilities.py)

In [ ]:
"""
Automatic Cryptocurrency trading using Deep RL
Nick Kaparinos
2022
"""

import random
import wandb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from pickle import dump, load
from typing import Callable, Optional, Tuple, List, Sequence
import bz2


def make_learning_curve(project: str, model_name: str, previous_runs: Sequence[str], log_dir: str, windows: List[int],
                        continue_from_previous_run: Optional[bool] = False) -> None:
    """ Make learning curve and log to wandb """
    api = wandb.Api()
    run = api.run(f'saoudioussama13-talon-one/{project}/{model_name}')
    history = run.scan_history()
    episode_rewards = get_episode_rewards(history)

    # Download previous runs data
    for previous_run in reversed(previous_runs):
        if continue_from_previous_run:
            run = api.run('saoudioussama13-talon-one/' + project + '/' + previous_run)
            history = run.scan_history()
            episode_rewards_temp = get_episode_rewards(history)
            episode_rewards = pd.concat([episode_rewards_temp, episode_rewards], axis=0).reset_index(drop=True)

    # Learning Curve
    for window in windows:
        learning_curve(episode_rewards=episode_rewards, log_dir=log_dir, window=window)


def learning_curve(episode_rewards: pd.DataFrame, log_dir: str, window: int) -> None:
    # Calculate rolling window metrics
    rolling_average = episode_rewards.rolling(window=window, min_periods=1).mean().dropna()
    rolling_max = episode_rewards.rolling(window=window, min_periods=1).max().dropna()
    rolling_min = episode_rewards.rolling(window=window, min_periods=1).min().dropna()

    # Change column name
    rolling_average.columns = ['Average Reward']
    rolling_max.columns = ['Max Reward']
    rolling_min.columns = ['Min Reward']
    rolling_data = pd.concat([rolling_average, rolling_max, rolling_min], axis=1)

    # Plot
    sns.set()
    plt.figure(0, dpi=200)
    plt.clf()
    ax = sns.lineplot(data=rolling_data)
    ax.fill_between(rolling_average.index, rolling_min.iloc[:, 0], rolling_max.iloc[:, 0], alpha=0.2)
    ax.set_title('Learning Curve', fontsize=16)
    ax.set_ylabel('Reward')
    ax.set_xlabel('Episodes')

    img_path = f'{log_dir}learning_curve{window}.png'
    plt.savefig(img_path, dpi=200)

    # Log figure
    image = plt.imread(img_path)
    wandb.log({f'Learning_Curve_{window}': [wandb.Image(image, caption="Learning_curve")]})


def get_episode_rewards(history: Sequence) -> pd.DataFrame:
    """ Get episode rewards from wandb scan history """
    episode_rewards_temp = []
    for i in history:
        if 'train/reward' in i.keys():
            episode_rewards_temp.append(i['train/reward'])
    return pd.DataFrame(data=episode_rewards_temp, columns=['Rewards'])


def build_test_fn(policy, optim, log_dir: str, model_name: str, train_collector,
                  save_train_buffer: Optional[bool] = True, model: Optional[str] = 'DQN') -> Callable:
    """ Build custom test function """

    def custom_test_fn(epoch, env_step):
        # Save agent
        print(f"Epoch = {epoch}")
        if model == 'DQN' or model == 'PPO':
            torch.save({'model': policy.state_dict(), 'optim': optim.state_dict()},
                       log_dir + model_name + f'_epoch{epoch}.pth')
        else:
            torch.save({'model': policy.state_dict(), 'actor_optim': optim['actor_optim'].state_dict(),
                        'critic1_optim': optim['critic1_optim'].state_dict(),
                        'critic2_optim': optim['critic1_optim'].state_dict()},
                       log_dir + model_name + f'_epoch{epoch}.pth')
        if save_train_buffer:
            with bz2.BZ2File(log_dir + f'epoch{epoch}_train_buffer' + '.pbz2', 'w') as f:
                dump(train_collector.buffer, f)

    return custom_test_fn


def build_epsilon_schedule(policy, max_epsilon: Optional[float] = 0.5, min_epsilon: Optional[float] = 0.0,
                           num_episodes_decay: Optional[int] = 10000) -> Callable:
    """ Build epsilon schedule function """

    def custom_epsilon_schedule(epoch, env_step):
        decay_step = (max_epsilon - min_epsilon) / num_episodes_decay
        current_epsilon = max_epsilon - env_step * decay_step
        if current_epsilon < min_epsilon:
            current_epsilon = min_epsilon
        policy.set_eps(current_epsilon)
        wandb.log({"train/env_step": env_step, 'epsilon': current_epsilon})

    return custom_epsilon_schedule


def load_previous_run(previous_runs: List[str], latest_run_epoch: int, policy, train_collector, env_id: str) -> None:
    """ Load model, optimizers and buffer from previous run"""
    assert latest_run_epoch > 0
    latest_run = previous_runs[-1]
    checkpoint = torch.load(f'logs/trading/{env_id[:-3]}/{latest_run}/{latest_run}_epoch{latest_run_epoch}.pth')
    policy.load_state_dict(checkpoint['model'])
    policy.optim.load_state_dict(checkpoint['optim'])
    with bz2.BZ2File(f'logs/trading/{env_id[:-3]}/{latest_run}/epoch{latest_run_epoch}_train_buffer.pbz2') as f:
        train_collector.buffer = load(f)


def save_dict_to_txt(dictionary: dict, path: str, txt_name: Optional[str] = 'hyperparameter_dict') -> None:
    """ Save dictionary as txt file """
    with open(f'{path}/{txt_name}.txt', 'w') as f:
        f.write(str(dictionary))


def save_list_to_txt(my_list: list, name: Optional[str] = 'epoch_distribution.txt') -> None:
    """ Save list as txt file """
    with open(name, 'w') as f:
        for item in my_list:
            f.write("%s\n" % item)


def set_all_seeds(seed: Optional[int] = 0) -> None:
    """ Set all seeds """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True  # noqa

print('utilities loaded.')


## Training

In [ ]:
import time
import torch.optim
import tianshou as ts
from tianshou.policy import DQNPolicy
from tianshou.utils import WandbLogger
from tianshou.data import PrioritizedVectorReplayBuffer as PVRB, VectorReplayBuffer as VRB
from torch.utils.tensorboard import SummaryWriter

start = time.perf_counter()
seed = 0
set_all_seeds(seed=seed)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

env_id = 'TradeEnv-v0'
test_env_id = 'TestTradeEnv-v0'

num_envs = 1
continue_from_previous_run, latest_run_epoch = False, -1
previous_runs = ['']
early_stop_patience = 4  # stop if no improvement for this many epochs
trainer_hyperparameters = {'max_epoch': 15, 'step_per_epoch': 150_000, 'step_per_collect': 100,
                           'episode_per_test': 25, 'batch_size': 128, 'update_per_step': 0.1}
policy_hyperparameters = {'discount_factor': 0.99, 'estimation_step': 1, 'target_update_freq': 20,
                          'is_double': True}
epsilon_schedule_hyperparameters = {'max_epsilon': 0.6, 'min_epsilon': 0.0,
                                     # Fixed at 6-epoch decay rate so epsilon reaches 0 by ~epoch 5-6
                                     'num_episodes_decay': int(trainer_hyperparameters['step_per_epoch'] * 6 * 0.9)}
replay_buffer_hyperparameters = {'total_size': 500_000, 'buffer_num': num_envs, 'alpha': 0.7, 'beta': 0.5}
net_hyperparameters = {'encoder_type': 'LSTM', 'n_neurons': 128, 'encoder_n_linear_layers': 2,
                       'q_n_linear_layers': 2, 'v_n_linear_layers': 2, 'n_posmlp_layers': 2,
                       'n_head_layers': 1, 'n_attention_blocks': 1, 'n_cnn_layers': 2, 'dueling': True}
env_kwargs = {
    'crypto_files': ['Data/ADAUSDT_minutes.hdf5', 'Data/BTCUSDT_minutes.hdf5',
                     'Data/ETHUSDT_minutes.hdf5', 'Data/LTCUSDT_minutes.hdf5'],
    'timeseries_step': '1h', 'max_episode_steps': max_episode_steps, 'n_previous_timesteps': 23}
misq_dict = {'learning_rate': 1e-4, 'seed': seed, 'use_prioritised_replay_buffer': True, 'optimizer': 'Adam',
             'continue_from_previous_run': continue_from_previous_run, 'algorithm': 'DQN',
             'latest_run_epoch': latest_run_epoch, 'previous_runs': previous_runs,
             'early_stop_patience': early_stop_patience,
             'normalisation': 'rolling_zscore'}
config = dict(policy_hyperparameters, **trainer_hyperparameters, **net_hyperparameters,
              **epsilon_schedule_hyperparameters, **replay_buffer_hyperparameters, **env_kwargs, **misq_dict)

model_name = f'{config["algorithm"]}_rolling_zscore_{time.strftime("%d_%b_%Y_%H_%M_%S", time.localtime())}'
log_dir = f'logs/trading/{env_id[:-3]}/{model_name}/'
makedirs(log_dir, exist_ok=True)
project = 'Gym-' + env_id[:-3]
logger = WandbLogger(train_interval=1, save_interval=1, project=project,
                     name=model_name, run_id=model_name, config=config)
logger.load(SummaryWriter(log_dir))

train_envs = ts.env.SubprocVectorEnv([lambda: gym.make(env_id, **env_kwargs) for _ in range(num_envs)])
test_envs = ts.env.SubprocVectorEnv(
    [lambda: gym.make(test_env_id, log_dir=log_dir,
                      num_test_episodes=config['episode_per_test'], **env_kwargs)
     for _ in range(num_envs)])
train_envs.seed(seed)
test_envs.seed(seed)
env = gym.make(env_id, **env_kwargs)

state_shape = env.observation_space.shape or env.observation_space.n
action_shape = env.action_space.shape or env.action_space.n
net = MLP(state_shape, action_shape, env.n_features, env.n_previous_timesteps, env.n_timeseries,
          **net_hyperparameters, device=device).to(device)
optim = torch.optim.Adam(net.parameters(), lr=config['learning_rate'])
policy = DQNPolicy(net, optim, **policy_hyperparameters)

train_collector = ts.data.Collector(policy, train_envs, PVRB(**replay_buffer_hyperparameters),
                                    exploration_noise=True)
test_collector = ts.data.Collector(policy, test_envs, exploration_noise=False)

if continue_from_previous_run:
    load_previous_run(previous_runs, latest_run_epoch, policy, train_collector, env_id)

_es_best_reward = float('-inf')
_es_no_improve = 0

policy.set_eps(epsilon_schedule_hyperparameters['max_epsilon'])
save_dict_to_txt(config, path=log_dir, txt_name='config')
train_fn = build_epsilon_schedule(policy=policy, **epsilon_schedule_hyperparameters)
_base_test_fn = build_test_fn(policy, optim, log_dir, model_name, train_collector, True, model=config['algorithm'])

_es_after_test = False

def test_fn(epoch, env_step):
    global _es_after_test
    _es_after_test = True
    _base_test_fn(epoch, env_step)

def stop_fn(mean_rewards):
    global _es_best_reward, _es_no_improve, _es_after_test
    if not _es_after_test:
        return False
    _es_after_test = False
    if mean_rewards > _es_best_reward:
        _es_best_reward = mean_rewards
        _es_no_improve = 0
        print(f'  [EarlyStopping] New best test reward: {mean_rewards:.4f}')
        return False
    _es_no_improve += 1
    print(f'  [EarlyStopping] No improvement for {_es_no_improve}/{early_stop_patience} epochs '
          f'(best: {_es_best_reward:.4f})')
    return _es_no_improve >= early_stop_patience

result = ts.trainer.offpolicy_trainer(policy, train_collector, test_collector, **trainer_hyperparameters,
                                      train_fn=train_fn, test_fn=test_fn, stop_fn=stop_fn, logger=logger)
print(f'Finished training! Duration {result["duration"]}')

windows = [25, 50]
make_learning_curve(project, model_name, previous_runs, log_dir, windows, continue_from_previous_run)
wandb.finish()
end = time.perf_counter()
print(f'Execution time = {end - start:.2f} second(s)')


## Download Results

In [ ]:
import shutil

shutil.make_archive('logs', 'zip', 'logs')
# On Kaggle the zip appears in the output panel automatically.
# On Colab uncomment the next two lines:
# from google.colab import files
# files.download('logs.zip')
print('logs.zip ready.')